# Building the synthetic data, and the test suite

`data/` is not shipped with this repository -- it is generated by this notebook (and, as a
fallback, by every other notebook's own setup cell, if it finds `data/` missing). Every value
in it is drawn from chosen statistical distributions (`numpy.random`, with hand-picked
parameters) -- none of it is fit to, sampled from, or otherwise derived from real data of any
kind; see `../src/scripts/make_synthetic_data.py`'s own docstring for exactly what structure
is planted and why. This notebook runs that script and
`../src/scripts/build_inputs.py` from scratch into a throwaway folder, writes `../data/` from
that if it does not already exist, and otherwise checks that a fresh rebuild is byte-identical
to what is already there -- i.e. that the data is actually reproducible from the code, not a
one-off file nobody can regenerate. It then runs the project's own unit tests
(`../src/tests/`).


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent
SRC_DIR = REPO_ROOT / "src"
SCRATCH = Path.cwd() / "_scratch_rebuild"
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir(parents=True)

env = os.environ.copy()
env["KMP_DUPLICATE_LIB_OK"] = "TRUE"
env["NUMBA_THREADING_LAYER"] = "workqueue"
env["PYTHONPATH"] = str(SRC_DIR)  # so a plain `python scripts/x.py` can "from metabo import ..."


def hide_local_paths(text):
    text = text.replace(str(REPO_ROOT), ".")
    return re.sub(r"(/home|/Users)/[^/]+", "~", text)


def run(*args):
    r = subprocess.run([sys.executable, *args], cwd=SRC_DIR, env=env,
                       capture_output=True, text=True)
    print(hide_local_paths(r.stdout)[-2000:])
    if r.returncode != 0:
        print(hide_local_paths(r.stderr)[-2000:])
    r.check_returncode()


## 1. `scripts/make_synthetic_data.py`

Writes the raw source file (semicolon separated, decimal comma, below-detection values
left empty) and the target file, with known structure planted (see the script's own
docstring for exactly what and why). The generator is seeded, so this is deterministic.


In [2]:
run("scripts/make_synthetic_data.py", "--outdir", str(SCRATCH / "data"), "--force")


wrote 1180 samples x 92 metabolites and 14 outcomes to ./notebooks/_scratch_rebuild/data
events: {'AD 6 yr ever': 41, 'Asthma 6 yr ever': 35, 'atopy_temp': 78, 'sens_s_ig_e_any': 71, 'sens_inhal': 47, 'sens_food': 36, 'ar_6y': 28, 'fetal-Antibiotics during pregnancy': 82, 'fetal-Antibiotics during birth': 61, 'fetal-Delivery': 62, 'mother_smoking': 24, 'fishoil': 133, 'vitd': 93, 'sex': 131}



## 2. `scripts/build_inputs.py`: the per-timepoint, log10, below-detection-filled input matrices

In [3]:
run("scripts/build_inputs.py",
    "--source", str(SCRATCH / "data" / "datacorr_6m6y18mMw1w24_final.csv"),
    "--outdir", str(SCRATCH / "data"))


660 child samples, 92 metabolites, 10.7% below detection
metabolites_6m_for_ml.csv: 240 children
metabolites_18m_for_ml.csv: 220 children
metabolites_6y_for_ml.csv: 200 children



## 3. Write (or check) `data/`

The two scripts above always write the real pipeline's own filenames
(`datacorr_6m6y18mMw1w24_final.csv` etc.) -- they have no option to name their output
differently, so each one is renamed (`synthetic_*`) on the way into `data/`, purely so it
reads as obviously synthetic at a glance. If `data/` does not exist yet, this is what
creates it; if it already does, this instead checks that a fresh rebuild reproduces it
exactly, byte for byte.


In [4]:
DATA_DIR = REPO_ROOT / "data"
RENAMED = {
    "datacorr_6m6y18mMw1w24_final.csv": "synthetic_raw_metabolome.csv",
    "target_for_metabolites_ml.csv": "synthetic_targets.csv",
    "metabolites_6m_for_ml.csv": "synthetic_metabolites_6m.csv",
    "metabolites_18m_for_ml.csv": "synthetic_metabolites_18m.csv",
    "metabolites_6y_for_ml.csv": "synthetic_metabolites_6y.csv",
}

if not all((DATA_DIR / new).exists() for new in RENAMED.values()):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for old, new in RENAMED.items():
        shutil.copy(SCRATCH / "data" / old, DATA_DIR / new)
    print(f"wrote {DATA_DIR} (did not exist yet)")
else:
    all_match = True
    for built_name, checked_in_name in RENAMED.items():
        a = pd.read_csv(SCRATCH / "data" / built_name, sep=";" if built_name.endswith("_final.csv") else ",",
                         decimal="," if built_name.endswith("_final.csv") else ".")
        b = pd.read_csv(DATA_DIR / checked_in_name, sep=";" if built_name.endswith("_final.csv") else ",",
                         decimal="," if built_name.endswith("_final.csv") else ".")
        same = a.equals(b)
        all_match &= same
        print(f"{checked_in_name}: {'identical' if same else 'DIFFERS'} ({a.shape})")
    print("\nAll files reproduce data/ exactly:" if all_match else "\nMISMATCH found:", all_match)


synthetic_raw_metabolome.csv: identical ((1180, 98))
synthetic_targets.csv: identical ((260, 15))
synthetic_metabolites_6m.csv: identical ((240, 93))
synthetic_metabolites_18m.csv: identical ((220, 93))
synthetic_metabolites_6y.csv: identical ((200, 93))

All files reproduce data/ exactly: True


## 4. The test suite

`src/tests/` holds known-answer unit tests for the mappings used throughout (PCA
attribution reproduces the classifier's own decision function, the opposite-sign case,
BH-FDR correctness). These do not need any data at all.


In [5]:
run("-m", "pytest", "tests", "-v", "--no-header")


============================= test session starts ==============================
collecting ... collected 6 items

tests/test_backprojection.py::test_opposite_signs_cancel PASSED          [ 16%]
tests/test_backprojection.py::test_pca_weights_reproduce_classifier PASSED [ 33%]
tests/test_backprojection.py::test_negative_importance_is_noise PASSED   [ 50%]
tests/test_stats.py::test_bh_known_answer PASSED                         [ 66%]
tests/test_stats.py::test_bh_preserves_input_order PASSED                [ 83%]
tests/test_stats.py::test_bh_clips_to_one PASSED                         [100%]

============================== 6 passed in 2.63s ===============================



In [6]:
shutil.rmtree(SCRATCH, ignore_errors=True)
print("cleaned up the scratch rebuild folder")


cleaned up the scratch rebuild folder


## Next steps

- Regenerate `../data/` itself (a new seed, a different sample size) by editing
  `scripts/make_synthetic_data.py`'s constants, rerunning the two commands above with
  `--outdir ../data`, then renaming the five output files as `RENAMED` above maps them.
